In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully!')

In [ ]:
DATASET_PATH = 'UTKFace'   
IMG_SIZE     = 64          
MAX_SAMPLES  = 5000      

images, genders, ages = [], [], []

files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.jpg')]
if MAX_SAMPLES:
    files = files[:MAX_SAMPLES]

print(f'Loading {len(files)} images from UTKFace...')

for fname in tqdm(files):
    try:
        parts  = fname.split('_')
        age    = int(parts[0])
        gender = int(parts[1]) 

        if age < 1 or age > 90:
            continue

        img_path = os.path.join(DATASET_PATH, fname)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        images.append(img)
        genders.append(gender)
        ages.append(age)

    except Exception:
        continue

# Convert to numpy arrays
X = np.array(images, dtype='float32') / 255.0  
y_gender = np.array(genders)
y_age    = np.array(ages)

def age_to_group(age):
    if age <= 20:  return 0
    if age <= 35:  return 1
    if age <= 55:  return 2
    return 3

y_age_group = np.array([age_to_group(a) for a in y_age])

print(f'\nDataset loaded successfully!')
print(f'Total samples  : {len(X)}')
print(f'Image shape    : {X[0].shape}  (64x64 RGB)')
print(f'Male   (0)     : {(y_gender==0).sum()}')
print(f'Female (1)     : {(y_gender==1).sum()}')
print(f'Age range      : {y_age.min()} – {y_age.max()} years')
unique, counts = np.unique(y_age_group, return_counts=True)
labels = ['0-20','21-35','36-55','56+']
for g, c in zip(unique, counts):
    print(f'  Age group {labels[g]:>5} : {c} samples')

In [ ]:
gender_label = {0: 'Male', 1: 'Female'}
age_grp_label = {0: '0-20', 1: '21-35', 2: '36-55', 3: '56+'}

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle('UTKFace Dataset — Sample Images', fontsize=13, fontweight='bold', color='#1F4E79')

for i, ax in enumerate(axes.flat):
    ax.imshow(X[i])
    ax.set_title(f"{gender_label[y_gender[i]]}\nAge {y_age[i]}", fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# One-hot encode labels for Keras
y_gender_cat    = to_categorical(y_gender,    num_classes=2)
y_age_cat       = to_categorical(y_age_group, num_classes=4)

(
    X_train, X_test,
    yg_train, yg_test,
    ya_train, ya_test,
    yg_test_raw, _,
    ya_test_raw, _
) = train_test_split(
    X,
    y_gender_cat, y_age_cat,
    y_gender,     y_age_group,   
    test_size=0.20, random_state=42
)

print(f'Train samples : {X_train.shape[0]}')
print(f'Test  samples : {X_test.shape[0]}')
print(f'Input shape   : {X_train.shape[1:]}')


In [ ]:
def build_model(input_shape=(64, 64, 3)):
    inputs = layers.Input(shape=input_shape)

    # ── Shared CNN backbone ──────────────────────────────────
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    # ── Output head A: Gender ────────────────────────────────
    gender_out = layers.Dense(2, activation='softmax', name='gender')(x)

    # ── Output head B: Age Group ─────────────────────────────
    age_out = layers.Dense(4, activation='softmax', name='age_group')(x)

    model = models.Model(inputs=inputs, outputs=[gender_out, age_out])
    return model

model = build_model()

model.compile(
    optimizer='adam',
    loss={
        'gender':    'categorical_crossentropy',
        'age_group': 'categorical_crossentropy'
    },
    metrics={
        'gender':    'accuracy',
        'age_group': 'accuracy'
    }
)

model.summary()

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    {'gender': yg_train, 'age_group': ya_train},
    validation_data=(X_test, {'gender': yg_test, 'age_group': ya_test}),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

print('\nTraining complete!')

In [ ]:
results = model.evaluate(X_test, {'gender': yg_test, 'age_group': ya_test}, verbose=0)

metric_names = model.metrics_names
for name, val in zip(metric_names, results):
    print(f'{name:35s}: {val:.4f}')

yg_pred_prob, ya_pred_prob = model.predict(X_test, verbose=0)
yg_pred = np.argmax(yg_pred_prob, axis=1)
ya_pred = np.argmax(ya_pred_prob, axis=1)

yg_true = np.argmax(yg_test, axis=1) if getattr(yg_test, 'ndim', 1) > 1 else np.asarray(yg_test).reshape(-1)
ya_true = np.argmax(ya_test, axis=1) if getattr(ya_test, 'ndim', 1) > 1 else np.asarray(ya_test).reshape(-1)

gender_acc = (yg_pred == yg_true).mean() * 100
age_acc    = (ya_pred == ya_true).mean() * 100

print(f'\nGender Test Accuracy   : {gender_acc:.2f}%')
print(f'Age Group Test Accuracy: {age_acc:.2f}%')

gender_labels = list(range(yg_pred_prob.shape[1]))
age_labels = list(range(ya_pred_prob.shape[1]))

gender_names = ['Male', 'Female']
age_names = ['0-20', '21-35', '36-55', '56+']

if len(gender_names) != len(gender_labels):
    gender_names = [f'Class {i}' for i in gender_labels]
if len(age_names) != len(age_labels):
    age_names = [f'Class {i}' for i in age_labels]

print('\n--- Gender Classification Report ---')
print(classification_report(yg_true, yg_pred, labels=gender_labels, target_names=gender_names, zero_division=0))

print('--- Age Group Classification Report ---')
print(classification_report(ya_true, ya_pred, labels=age_labels, target_names=age_names, zero_division=0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Gender & Age Detection — CNN Results (UTKFace)', fontsize=14,
             fontweight='bold', color='#1F4E79')

axes[0,0].plot(history.history['gender_accuracy'],    color='#2E75B6', lw=2, label='Train')
axes[0,0].plot(history.history['val_gender_accuracy'],color='#C55A11', lw=2, label='Validation')
axes[0,0].set_title('Gender — Accuracy', fontweight='bold', color='#1F4E79')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

axes[0,1].plot(history.history['age_group_accuracy'],    color='#375623', lw=2, label='Train')
axes[0,1].plot(history.history['val_age_group_accuracy'],color='#C55A11', lw=2, label='Validation')
axes[0,1].set_title('Age Group — Accuracy', fontweight='bold', color='#1F4E79')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy')
axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

cm_g = confusion_matrix(yg_true, yg_pred, labels=gender_labels)
ConfusionMatrixDisplay(cm_g, display_labels=gender_names).plot(
    ax=axes[1,0], colorbar=False, cmap='Blues')
axes[1,0].set_title(f'Gender Confusion Matrix  (Acc: {gender_acc:.1f}%)',
                    fontweight='bold', color='#1F4E79')

cm_a = confusion_matrix(ya_true, ya_pred, labels=age_labels)
ConfusionMatrixDisplay(cm_a, display_labels=age_names).plot(
    ax=axes[1,1], colorbar=False, cmap='Greens')
axes[1,1].set_title(f'Age Group Confusion Matrix  (Acc: {age_acc:.1f}%)',
                    fontweight='bold', color='#1F4E79')

plt.tight_layout()
plt.savefig('results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
gender_map  = {0: 'Male', 1: 'Female'}
age_grp_map = {0: '0-20', 1: '21-35', 2: '36-55', 3: '56+'}

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
fig.suptitle('Sample Predictions on Test Images', fontsize=13,
             fontweight='bold', color='#1F4E79')

for i, ax in enumerate(axes.flat):
    img     = X_test[i]
    true_g  = gender_map[int(yg_true[i])]
    true_ag = age_grp_map.get(int(ya_true[i]), f'Class {int(ya_true[i])}')
    pred_g  = gender_map[int(yg_pred[i])]
    pred_ag = age_grp_map.get(int(ya_pred[i]), f'Class {int(ya_pred[i])}')
    correct = (int(yg_true[i]) == int(yg_pred[i]))

    ax.imshow(img)
    ax.set_title(
        f'True: {true_g} | {true_ag}\nPred: {pred_g} | {pred_ag}',
        fontsize=8,
        color='green' if correct else 'red'
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model.save('gender_age_model.keras')
print('Model saved as gender_age_model.keras')